In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/results"

silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/results"

In [0]:
results_df = (
    spark.read.format("delta")
        .load(bronze_path)
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
display(results_df)

In [0]:
results_selected_df = results_df.select(
    "season",
    "round",
    "constructorId",
    "driverId",
    "date",
    "raceName",
    "grid",
    "laps",
    "number",
    "points",
    "position",
    "positionText",
    "status",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
)

In [0]:
results_renamed_df = (
    results_selected_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "positionText": "finish_position_text",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "finish_position"
        })
)

In [0]:
display(results_renamed_df)

In [0]:
results_valid_df = (
    results_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)

In [0]:
display(
    results_renamed_df.count() - results_valid_df.count()
)

In [0]:
results_distinct_df = results_valid_df.dropDuplicates([
    "season",
    "round",
    "constructor_id",
    "driver_id"
])

In [0]:
display(
    results_valid_df.count() - results_distinct_df.count()
)

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
results_final_df = (
    results_final_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
display(results_final_df)

In [0]:
if not DeltaTable.isDeltaTable(spark, silver_path):
    (
        results_final_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
    )
else:
    silver_delta = DeltaTable.forPath(spark, silver_path)

    (
        silver_delta.alias("t")
            .merge(
                results_final_df.alias("s"),
                """
                t.season = s.season
                AND t.round = s.round
                AND t.constructor_id = s.constructor_id
                AND t.driver_id = s.driver_id
                """
            )
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set={
                    "race_date": "s.race_date",
                    "race_name": "s.race_name",
                    "grid_position": "s.grid_position",
                    "completed_laps": "s.completed_laps",
                    "car_number": "s.car_number",
                    "points": "s.points",
                    "finish_position": "s.finish_position",
                    "finish_position_text": "s.finish_position_text",
                    "status": "s.status",
                    "ingestion_timestamp": "s.ingestion_timestamp",
                    "source_file": "s.source_file",
                    "batch_id": "s.batch_id",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
results_silver_df = spark.read.format("delta").load(silver_path)

display(results_silver_df)